# BCU AI Hackathon 2026 — RAG Multiple-Choice Pipeline
**Team: MACINTOSH**

This notebook answers the 100 multiple-choice questions in `questions_100.csv`.

### Why this design
The questions are **long-tail, Wikipedia-sourced facts** (tower heights, species ranges, obscure people/albums). A ≤8B model knows almost none of them from memory, so this is a **retrieval + reading-comprehension** problem, not a reasoning one. The answer is almost always sitting *verbatim* in one Wikipedia article. The pipeline therefore prioritises **retrieval quality** over everything else:

1. **Entity extraction** — find the searchable subject of each question.
2. **Wikipedia-first retrieval** — fetch the *full* article (not short snippets), with a DuckDuckGo fallback.
3. **Cross-encoder reranking** (`bge-reranker-v2-m3`) — keep only the passages that actually answer the question.
4. **Answer extraction** — self-consistency majority vote **with option-order shuffling** to cancel the well-known A/position bias of small models.
5. **Evidence verification** — tie-break / fallback by checking which option's claim the evidence best supports.
6. **Never blank** — assuming pure accuracy (no penalty for wrong), every row gets a committed A–E guess. *Confirm with the organiser whether wrong answers are penalised; if so, flip `ALLOW_UNKNOWN`.*

### Model rule (≤ 8B params)
Default backend is **Ollama + Qwen2.5-7B-Instruct (~7.6B)** — safely under the cap and excellent at evidence-grounded comprehension. State the exact model + size in your slides/README, as the brief requires. (Note: Qwen3-8B is **8.2B** and would exceed the limit.)

### How to run
1. Install deps (next cell) and Ollama from https://ollama.com, then `ollama pull qwen2.5:7b-instruct`.
2. Put `questions_100.csv` next to this notebook (or set `QUESTIONS_FILE`).
3. Run the cells top to bottom. Use the smoke-test cell first, then the full run.


In [7]:
!pip3 install  pandas requests ddgs sentence-transformers ftfy

  Using cached pandas-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached numpy-2.4.6-cp313-cp313-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached charset_normalizer-3.4.7-cp313-cp313-macosx_10_13_universal2.whl.metadata (40 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.7 kB)
Using cached pandas-3.0.3-cp313-cp313-macosx_11_0_arm64.whl (9.9 MB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached char

In [11]:
import os
import random

# ---------- Team / IO ----------
TEAM_NAME      = "MACINTOSH"
QUESTIONS_FILE = "questions_100.csv"                 # path to official questions
OUTPUT_FILE    = f"{TEAM_NAME}_submission.csv"       # REQUIRED output name
EVIDENCE_LOG   = f"{TEAM_NAME}_evidence_log.jsonl"   # debug / reproducibility log
CACHE_FILE     = "retrieval_cache.json"              # speeds up re-runs

# ---------- LLM backend ----------
# "ollama" -> local, runs on your Mac (recommended).  "openai" -> any OpenAI-compatible API.
LLM_BACKEND  = "ollama"

# Model MUST be <= 8B. Qwen2.5-7B-Instruct (~7.6B) is safely under the cap.
OLLAMA_MODEL = "qwen2.5:7b-instruct"
OLLAMA_URL   = "http://localhost:11434/api/chat"

# Used only when LLM_BACKEND == "openai" (set OPENAI_API_KEY in your environment)
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.groq.com/openai/v1")
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")
OPENAI_MODEL    = os.environ.get("OPENAI_MODEL", "llama-3.1-8b-instant")  # ~8B, under cap

# ---------- Retrieval ----------
USE_WIKIPEDIA    = True
USE_DDG          = True
USE_RERANKER     = True
RERANKER_MODEL   = "BAAI/bge-reranker-v2-m3"   # 278M, multilingual, runs on CPU
MAX_WIKI_ARTICLES = 2
DDG_MAX_RESULTS   = 4
TOP_K_PASSAGES    = 5
CHUNK_WORDS       = 130
CHUNK_OVERLAP     = 30

# ---------- Answer method ----------
N_SELF_CONSISTENCY = 5      # samples per question (majority vote + option shuffle)
SC_TEMPERATURE     = 0.7
CONFIDENCE_FLOOR   = 0.6    # below this, lean on evidence verification

# ---------- Scoring policy ----------
ALLOW_UNKNOWN = False       # assume NO penalty for wrong -> always guess

OPTION_KEYS  = ["A", "B", "C", "D", "E"]
HTTP_HEADERS = {"User-Agent": f"BCU-Hackathon-{TEAM_NAME}/1.0 (educational project)"}

random.seed(42)
print("Config loaded. Backend:", LLM_BACKEND, "| Model:", OLLAMA_MODEL if LLM_BACKEND=="ollama" else OPENAI_MODEL)

Config loaded. Backend: ollama | Model: qwen2.5:7b-instruct


## 1. Load questions (with mojibake repair)
A few rows contain UTF-8 mojibake (e.g. `BÃª Kontrol` → `Bê Kontrol`). We repair text on load so retrieval queries are clean.

In [13]:
import pandas as pd

try:
    import ftfy
    HAVE_FTFY = True
except ImportError:
    HAVE_FTFY = False

def fix_text(s):
    if not isinstance(s, str):
        return s
    if HAVE_FTFY:
        try:
            return ftfy.fix_text(s)
        except Exception:
            pass
    try:
        repaired = s.encode("latin-1").decode("utf-8")
        if repaired.count("\ufffd") <= s.count("\ufffd"):
            return repaired
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    return s

def load_questions(path):
    df = pd.read_csv(path)
    required = {"question_no", "question"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    for col in ["question"] + OPTION_KEYS:
        if col in df.columns:
            df[col] = df[col].apply(fix_text)
    return df

questions = load_questions(QUESTIONS_FILE)
print(f"Loaded {len(questions)} questions")
questions.head(3)

Loaded 100 questions


,question_no,question,A,B,C,D,E
0,1,What are the core capabilities of United State...,Electronic warfare and intelligence analysis.,"Direct action, special reconnaissance and fore...",Airborne operations and cyber warfare.,Maritime interdiction and unconventional warfare.,Counterintelligence and emergency medical resp...
1,2,In which war did General Sir George Anson serv...,Crimean War,War of 1812,American Revolutionary War,Napoleonic Wars,French Revolution
2,3,What was P. Padmarajan's contribution to Malay...,P. Padmarajan was a noted film critic who revo...,"P. Padmarajan was an Indian film maker, screen...",P. Padmarajan was a pioneer in introducing adv...,P. Padmarajan was a renowned cinematographer k...,P. Padmarajan popularized the use of complex n...


## 2. LLM backend (Ollama / OpenAI-compatible)
A single `llm_chat()` call abstracts the backend. Switch with `LLM_BACKEND` in the config. Run the health-check below before going further.

In [14]:
import json
import time
import requests

_openai_client = None
def _get_openai_client():
    global _openai_client
    if _openai_client is None:
        from openai import OpenAI
        _openai_client = OpenAI(base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY)
    return _openai_client

def llm_chat(messages, temperature=0.0, max_tokens=512, seed=None):
    """Unified chat call -> returns assistant text."""
    if LLM_BACKEND == "ollama":
        options = {"temperature": float(temperature), "num_predict": int(max_tokens)}
        if seed is not None:
            options["seed"] = int(seed)
        payload = {"model": OLLAMA_MODEL, "messages": messages, "stream": False, "options": options}
        r = requests.post(OLLAMA_URL, json=payload, timeout=180)
        r.raise_for_status()
        return r.json()["message"]["content"]
    elif LLM_BACKEND == "openai":
        client = _get_openai_client()
        kwargs = {"model": OPENAI_MODEL, "messages": messages,
                  "temperature": float(temperature), "max_tokens": int(max_tokens)}
        if seed is not None:
            kwargs["seed"] = int(seed)
        r = client.chat.completions.create(**kwargs)
        return r.choices[0].message.content
    raise ValueError(f"Unknown LLM_BACKEND: {LLM_BACKEND}")

def llm_healthcheck():
    try:
        out = llm_chat([{"role": "user", "content": "Reply with the single word: OK"}], max_tokens=5)
        print(f"LLM backend '{LLM_BACKEND}' OK -> {out!r}")
        return True
    except Exception as e:
        print(f"LLM backend '{LLM_BACKEND}' NOT reachable: {e}")
        print("Ollama users: run `ollama serve` and `ollama pull qwen2.5:7b-instruct`.")
        return False

llm_healthcheck()

LLM backend 'ollama' OK -> 'OK'


True

## 3. Entity extraction & query building
We do **not** dump the question + all 5 options into one query (five different years/places poison the search). Instead we extract the subject entity and build a few focused queries.

In [31]:
import re

STOPWORDS = set("what when where which who whom whose why how is are was were the a an of in on at to for and or about does did".split())

def get_options(row):
    opts = {}
    for k in OPTION_KEYS:
        v = row.get(k, "")
        if pd.notna(v) and str(v).strip():
            opts[k] = str(v).strip()
    return opts

def heuristic_entity(question):
    quoted = re.findall(r"[\"\u201c]([^\"\u201d]{2,})[\"\u201d]", question)
    caps = re.findall(r"\b([A-Z][\w.\-]+(?:\s+[A-Z0-9][\w.\-]+)*)\b", question)
    caps = [c for c in caps if c.lower() not in STOPWORDS and len(c) > 2]
    parts = quoted + caps
    seen, out = set(), []
    for p in parts:
        if p not in seen:
            seen.add(p); out.append(p)
    return " ".join(out[:3]) if out else question

def llm_entity(question):
    prompt = ("Extract the main searchable subject of this question as it would appear as a "
              "Wikipedia article title. Reply with ONLY the subject, no extra words.\n\n"
              f"Question: {question}\nSubject:")
    try:
        out = llm_chat([{"role": "user", "content": prompt}], temperature=0.0, max_tokens=32)
        out = out.strip().splitlines()[0].strip().strip('".')
        if 2 <= len(out) <= 80:
            return out
    except Exception as e:
        print(f"  [entity] LLM failed, heuristic used: {e}")
    return heuristic_entity(question)

def build_queries(row, entity):
    question = str(row.get("question", "")).strip()
    q_terms = [w for w in re.findall(r"[A-Za-z][A-Za-z\-]+", question) if w.lower() not in STOPWORDS]
    tail = " ".join(q_terms[-4:])
    candidates = [entity, f"{entity} {tail}".strip(), question]
    seen, out = set(), []
    for q in candidates:
        q = q.strip()
        if q and q.lower() not in seen:
            seen.add(q.lower()); out.append(q)
    return out

## 4. Retrieval — Wikipedia (full article) + DuckDuckGo fallback
We fetch the **full plain-text article** via the MediaWiki API, not just a snippet, because the answer is often a single specific fact buried in the body. Results are cached to disk for fast re-runs.

In [32]:
from pathlib import Path

try:
    from ddgs import DDGS
    DDGS_AVAILABLE = True
except ImportError:
    try:
        from duckduckgo_search import DDGS
        DDGS_AVAILABLE = True
    except ImportError:
        DDGS_AVAILABLE = False

if Path(CACHE_FILE).exists():
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        _CACHE = json.load(f)
else:
    _CACHE = {}

def _save_cache():
    with open(CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(_CACHE, f)

WIKI_API = "https://en.wikipedia.org/w/api.php"

def wiki_search_titles(query, limit=3):
    key = f"wikisearch::{query}::{limit}"
    if key in _CACHE:
        return _CACHE[key]
    params = {"action": "opensearch", "search": query, "limit": limit, "namespace": 0, "format": "json"}
    titles = []
    try:
        r = requests.get(WIKI_API, params=params, headers=HTTP_HEADERS, timeout=30)
        r.raise_for_status()
        data = r.json()
        titles = data[1] if len(data) > 1 else []
    except Exception as e:
        print(f"  [wiki search] {query!r}: {e}")
    _CACHE[key] = titles
    return titles

def wiki_fetch_article(title):
    key = f"wikiarticle::{title}"
    if key in _CACHE:
        return _CACHE[key]
    params = {"action": "query", "prop": "extracts", "explaintext": 1,
              "redirects": 1, "titles": title, "format": "json"}
    text = ""
    try:
        r = requests.get(WIKI_API, params=params, headers=HTTP_HEADERS, timeout=30)
        r.raise_for_status()
        pages = r.json().get("query", {}).get("pages", {})
        for _, page in pages.items():
            text = page.get("extract", "") or ""
            break
    except Exception as e:
        print(f"  [wiki fetch] {title!r}: {e}")
    _CACHE[key] = text
    return text

def ddg_search(query, max_results=DDG_MAX_RESULTS):
    if not (USE_DDG and DDGS_AVAILABLE):
        return []
    key = f"ddg::{query}::{max_results}"
    if key in _CACHE:
        return _CACHE[key]
    hits = []
    try:
        with DDGS() as ddgs:
            for item in ddgs.text(query, max_results=max_results):
                hits.append({"title": item.get("title", ""), "snippet": item.get("body", ""),
                             "url": item.get("href", "")})
    except Exception as e:
        print(f"  [ddg] {query!r}: {e}")
    _CACHE[key] = hits
    time.sleep(0.4)
    return hits

def retrieve_evidence(row, queries):
    docs, seen_titles = [], set()
    if USE_WIKIPEDIA:
        for q in queries:
            for title in wiki_search_titles(q, limit=MAX_WIKI_ARTICLES):
                if title in seen_titles:
                    continue
                seen_titles.add(title)
                text = wiki_fetch_article(title)
                if text:
                    docs.append({"source": "wikipedia", "title": title, "text": text})
            if len(seen_titles) >= MAX_WIKI_ARTICLES + 1:
                break
    if USE_DDG:
        for q in queries[:2]:
            for hit in ddg_search(q):
                if hit["snippet"]:
                    docs.append({"source": "ddg", "title": hit["title"],
                                 "text": hit["snippet"], "url": hit.get("url", "")})
    return docs

## 5. Chunk + rerank
Split articles into passages, then a **cross-encoder reranker** scores each passage against the question and we keep the top-k. This is the single biggest precision boost — vector/keyword similarity ≠ relevance. Falls back to keyword overlap if the reranker is disabled.

In [33]:
_reranker = None
def get_reranker():
    global _reranker
    if _reranker is None:
        from sentence_transformers import CrossEncoder
        print(f"Loading reranker {RERANKER_MODEL} (first run downloads weights)...")
        _reranker = CrossEncoder(RERANKER_MODEL)
    return _reranker

def chunk_text(text, n_words=CHUNK_WORDS, overlap=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    chunks, i, step = [], 0, max(1, n_words - overlap)
    while i < len(words):
        ch = " ".join(words[i:i + n_words]).strip()
        if ch:
            chunks.append(ch)
        i += step
    return chunks

def build_passages(docs):
    passages = []
    for d in docs:
        for ch in chunk_text(d["text"]):
            passages.append({"title": d.get("title", ""), "source": d["source"], "text": ch})
    return passages

def keyword_rank(query, passages, top_k):
    q = set(re.findall(r"[a-z0-9]+", query.lower()))
    scored = [(len(q & set(re.findall(r"[a-z0-9]+", p["text"].lower()))), p) for p in passages]
    scored.sort(key=lambda x: x[0], reverse=True)
    return [p for _, p in scored[:top_k]]

def rank_passages(query, passages, top_k=TOP_K_PASSAGES):
    if not passages:
        return []
    if not USE_RERANKER:
        return keyword_rank(query, passages, top_k)
    ce = get_reranker()
    scores = ce.predict([(query, p["text"]) for p in passages])
    for p, s in zip(passages, scores):
        p["score"] = float(s)
    passages.sort(key=lambda x: x["score"], reverse=True)
    return passages[:top_k]

## 6. Prompt building & answer parsing
A compact RAG prompt with one few-shot example to lock format. `parse_letter` is robust to `Answer: C`, `(C)`, etc. `clean_final` guarantees a valid A–E (never blank).

In [34]:
FEWSHOT = """Question:
What is the capital city of Australia?

Options:
A. Sydney
B. Melbourne
C. Canberra
D. Perth
E. Brisbane

Evidence:
- [Canberra] Canberra is the capital city of Australia, in the Australian Capital Territory.

Reasoning: The evidence states Canberra is the capital; Sydney and Melbourne are larger but not the capital.
Answer: C"""

def format_options(opts):
    return "\n".join(f"{k}. {v}" for k, v in opts.items())

def format_evidence(passages):
    if not passages:
        return "(no evidence retrieved)"
    return "\n".join(f"- [{p.get('title','')}] {p['text']}" for p in passages)

def build_prompt(question, opts, passages):
    letters = ", ".join(opts.keys())
    return f"""You answer a multiple-choice question using the evidence provided.
Read the evidence, reason in ONE short sentence, then give the best option.
Choose exactly one letter from: {letters}. Always commit to your single best guess.

{FEWSHOT}

Question:
{question}

Options:
{format_options(opts)}

Evidence:
{format_evidence(passages)}

Reasoning:"""

LETTER_RE = re.compile(r"\b([A-E])\b")

def parse_letter(text, valid_keys):
    if not text:
        return None
    m = re.search(r"answer\s*[:\-]?\s*\(?([A-E])\)?", text, flags=re.IGNORECASE)
    if m and m.group(1).upper() in valid_keys:
        return m.group(1).upper()
    found = [c for c in LETTER_RE.findall(text.upper()) if c in valid_keys]
    return found[-1] if found else None

def clean_final(answer):
    a = str(answer).strip().upper()[:1]
    if a in OPTION_KEYS:
        return a
    if ALLOW_UNKNOWN and str(answer).strip().lower() == "unknown":
        return "Unknown"
    return "A"  # deterministic last resort; never blank

## 7. Answer selection — self-consistency + option-shuffle debiasing + verification
Small models are biased toward option **A** / certain positions (accuracy can swing ~25% by where the answer sits). We sample several times, **shuffling the option order each time**, map each vote back to the original letter, and take the majority. Ties (or no usable output) are broken by **evidence verification**: which option's claim does the reranker say the evidence best supports.

In [35]:
from collections import Counter

def _shuffled_options(opts, rng):
    keys = list(opts.keys())
    rng.shuffle(keys)
    disp, disp_to_orig = {}, {}
    for new_label, orig_key in zip(OPTION_KEYS, keys):
        disp[new_label] = opts[orig_key]
        disp_to_orig[new_label] = orig_key
    return disp, disp_to_orig

def answer_self_consistency(question, opts, passages, n=N_SELF_CONSISTENCY):
    votes = Counter()
    rng = random.Random(12345)
    for i in range(n):
        disp_opts, disp_to_orig = _shuffled_options(opts, rng)
        prompt = build_prompt(question, disp_opts, passages)
        try:
            out = llm_chat([{"role": "user", "content": prompt}],
                           temperature=SC_TEMPERATURE, max_tokens=160, seed=1000 + i)
        except Exception as e:
            print(f"  [sc] sample {i} failed: {e}")
            continue
        disp_letter = parse_letter(out, set(disp_opts.keys()))
        if disp_letter is not None:
            votes[disp_to_orig[disp_letter]] += 1
    return votes

def verify_pick(question, opts, passages):
    """Which option's claim is best supported by the evidence?"""
    if not passages or not opts:
        return None, 0.0
    if USE_RERANKER:
        ce = get_reranker()
        keys = list(opts.keys())
        pairs = [(f"{question} {opts[k]}", p["text"]) for k in keys for p in passages]
        scores = ce.predict(pairs)
        best, i, n_p = {}, 0, len(passages)
        for k in keys:
            best[k] = float(max(scores[i:i + n_p]))
            i += n_p
        pick = max(best, key=best.get)
        return pick, best[pick]
    # keyword fallback
    best = {}
    for k, v in opts.items():
        qt = set(re.findall(r"[a-z0-9]+", (question + " " + v).lower()))
        best[k] = max((len(qt & set(re.findall(r"[a-z0-9]+", p["text"].lower()))) for p in passages), default=0)
    pick = max(best, key=best.get)
    return pick, float(best[pick])

def choose_answer(question, opts, passages):
    votes = answer_self_consistency(question, opts, passages)
    total = sum(votes.values())
    if total == 0:
        vpick, _ = verify_pick(question, opts, passages)
        if vpick:
            return vpick, 0.0, "verify_fallback"
        return list(opts.keys())[0], 0.0, "blind_fallback"
    top_n = votes.most_common(1)[0][1]
    leaders = [k for k, c in votes.items() if c == top_n]
    confidence = top_n / total
    if len(leaders) == 1 and confidence >= CONFIDENCE_FLOOR:
        return leaders[0], confidence, "self_consistency"
    # tie OR low confidence -> verify among the relevant options
    target = {k: opts[k] for k in leaders} if len(leaders) > 1 else opts
    vpick, _ = verify_pick(question, target, passages)
    if len(leaders) > 1:
        return (vpick or leaders[0]), confidence, "verify_tiebreak"
    # single low-confidence leader: keep it unless verification disagrees with strong evidence
    return leaders[0], confidence, "self_consistency_lowconf"

## 8. Per-question pipeline
Ties everything together with error handling so **no row ever fails or is left blank**, and logs evidence for reproducibility and your slides.

In [36]:
def answer_question(row):
    qno = row.get("question_no")
    question = str(row.get("question", "")).strip()
    opts = get_options(row)
    rec = {"question_no": int(qno) if pd.notna(qno) else None, "question": question}
    try:
        entity = llm_entity(question)
        queries = build_queries(row, entity)
        docs = retrieve_evidence(row, queries)
        passages = rank_passages(queries[0], build_passages(docs), top_k=TOP_K_PASSAGES)
        letter, conf, method = choose_answer(question, opts, passages)
        rec.update({
            "entity": entity, "queries": queries, "n_docs": len(docs),
            "top_passages": [p["text"][:200] for p in passages[:3]],
            "answer": letter, "confidence": round(conf, 3), "method": method,
        })
    except Exception as e:
        print(f"  [q{qno}] error: {e}")
        rec.update({"answer": (list(opts.keys())[0] if opts else "A"),
                    "confidence": 0.0, "method": "error_fallback"})
    return rec

## 9. Smoke test (first 3 questions)
Run this before the full set to confirm retrieval and answering work end to end.

In [37]:
for _, row in questions.head(3).iterrows():
    rec = answer_question(row)
    print(f"Q{rec['question_no']}: {rec['answer']}  ({rec.get('method')}, conf={rec.get('confidence')})")
    print(f"   entity : {rec.get('entity')}")
    print(f"   #docs  : {rec.get('n_docs')}")
    if rec.get("top_passages"):
        print(f"   top    : {rec['top_passages'][0][:160]}...")
    print()
_save_cache()

Loading reranker BAAI/bge-reranker-v2-m3 (first run downloads weights)...


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 12476.24it/s]


Q1: B  (self_consistency, conf=0.8)
   entity : United States Marine Forces Special Operations Command
   #docs  : 9
   top    : The United States Marine Forces Special Operations Command (MARSOC) is one of the four primary component commands (USASOC, USNSWC, AFSOC, MARSOC) of the United ...

Q2: D  (self_consistency, conf=1.0)
   entity : Peninsular War
   #docs  : 10
   top    : The Peninsular War (1808–1814) was fought in the Iberian Peninsula by the Iberian nations Spain and Portugal, along with the United Kingdom, against the invadin...

Q3: B  (self_consistency, conf=1.0)
   entity : P. Padmarajan
   #docs  : 10
   top    : Padmarajan Padmanabhan Pillai, better known as P. Padmarajan (23 May 1945 – 23 January 1991) was an Indian film maker, screenwriter and author who was known for...



## 10. Full run + export submission
Runs all 100 questions and writes `MACINTOSH_submission.csv` plus an evidence log. Tip: pass `limit=10` first to test a subset, then run the full set.

In [38]:
def run_pipeline(df, limit=None):
    rows = df.head(limit) if limit else df
    records = []
    for i, (_, row) in enumerate(rows.iterrows(), 1):
        qno = row.get("question_no", i)
        rec = answer_question(row)
        records.append(rec)
        print(f"[{i}/{len(rows)}] Q{qno} -> {rec['answer']} ({rec.get('method')})")
        if i % 10 == 0:
            _save_cache()
    _save_cache()
    return records

records = run_pipeline(questions)            # <- use run_pipeline(questions, limit=10) to test first

submission = pd.DataFrame(
    [{"question_no": r["question_no"], "answer": clean_final(r["answer"])} for r in records],
    columns=["question_no", "answer"],
)
submission.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved {OUTPUT_FILE} with {len(submission)} rows")

with open(EVIDENCE_LOG, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Saved {EVIDENCE_LOG}")

[1/100] Q1 -> B (self_consistency)
[2/100] Q2 -> D (self_consistency)
[3/100] Q3 -> B (self_consistency)
[4/100] Q4 -> C (self_consistency)
[5/100] Q5 -> D (self_consistency)
[6/100] Q6 -> A (self_consistency)
[7/100] Q7 -> E (self_consistency)
[8/100] Q8 -> A (self_consistency)
[9/100] Q9 -> C (self_consistency)
[10/100] Q10 -> B (self_consistency)
[11/100] Q11 -> A (self_consistency)
[12/100] Q12 -> B (self_consistency)
[13/100] Q13 -> C (self_consistency)
[14/100] Q14 -> B (self_consistency)
[15/100] Q15 -> E (self_consistency)
[16/100] Q16 -> E (self_consistency)
[17/100] Q17 -> C (self_consistency)
[18/100] Q18 -> E (self_consistency)
[19/100] Q19 -> E (self_consistency)
[20/100] Q20 -> E (self_consistency_lowconf)
[21/100] Q21 -> B (self_consistency)
[22/100] Q22 -> D (self_consistency)
[23/100] Q23 -> C (self_consistency)
[24/100] Q24 -> B (self_consistency)
[25/100] Q25 -> A (self_consistency)
[26/100] Q26 -> A (self_consistency)
[27/100] Q27 -> C (self_consistency)
[28/100] Q2

## 11. Validate submission format
Hard checks before you submit: correct columns, no blanks, only allowed answers.

In [39]:
sub = pd.read_csv(OUTPUT_FILE)
assert list(sub.columns) == ["question_no", "answer"], "Columns must be exactly question_no,answer"
allowed = set(OPTION_KEYS) | ({"Unknown"} if ALLOW_UNKNOWN else set())
blanks = int(sub["answer"].isna().sum())
bad = sub[~sub["answer"].isin(allowed)]
print(f"Rows: {len(sub)} | Blank: {blanks} | Invalid: {len(bad)}")
print("Answer distribution:\n", sub["answer"].value_counts().to_string())
assert blanks == 0 and len(bad) == 0, "Fix invalid/blank answers before submitting!"
print("\nSubmission is valid.")

Rows: 100 | Blank: 0 | Invalid: 0
Answer distribution:
 answer
B    28
A    22
C    19
E    17
D    14

Submission is valid.


## 12. Error analysis — spend remaining time wisely
Lists your lowest-confidence answers. Hand-check a few against Wikipedia; if a whole *category* of entity keeps failing retrieval, tweak `build_queries` for it.

In [40]:
low = sorted(records, key=lambda r: r.get("confidence", 0))[:15]
for r in low:
    print(f"Q{r['question_no']} -> {r['answer']}  (conf={r.get('confidence')}, {r.get('method')})")
    print(f"   {r['question'][:110]}")
    if r.get("top_passages"):
        print(f"   evidence: {r['top_passages'][0][:140]}...")
    print()

Q20 -> E  (conf=0.4, self_consistency_lowconf)
   In which events did Konrad Sebastian Winkler compete at the 1924 Summer Olympics?
   evidence: Konrad Sebastian Winkler (20 January 1882 – 16 January 1962) was a Polish fencer. [1] He competed in the individual foil and team sabre at t...

Q31 -> C  (conf=0.4, verify_tiebreak)
   Which of the following statements accurately describes the production details of the song "Take Me Over" by Pe
   evidence: "Take Me Over" is a song by Australian electronic music duo Peking Duk. The song features vocals from SAFIA. It was produced by Adam Hyde an...

Q34 -> B  (conf=0.4, self_consistency_lowconf)
   What is the key structural motif of starch?
   evidence: In summary, the structural aspects of starch, including its molecular structure, molecular and glucose residue conformations, and hydrogen b...

Q47 -> C  (conf=0.4, verify_tiebreak)
   Which of the following statements accurately categorizes the monarchs, a family of passerine birds, based o